Task 5: Per-Tensor and Per-Channel Quantization

Objective: Compare per-tensor and per-output-channel symmetric quantization for convolution weights. Understand why per-channel quantization preserves accuracy better when channels have different value ranges.

In [1]:
import numpy as np
import pandas as pd

In [2]:
# creating tensor that mimic conv layer weights (as said in assignment)
np.random.seed(42)

conv_weights = np.random.randn(8, 3, 3, 3).astype(np.float32)

# creating imbalance
conv_weights[0] *= 0.1 # very small range
conv_weights[1] *= 0.5 # small range
conv_weights[6] *= 5.0 # large range
conv_weights[7] *= 10.0 # very large range

print(conv_weights.shape)

(8, 3, 3, 3)


## Part A: Per-Tensor Symmetric Quantization

In [3]:
def per_tensor_quantize(tensor):
    
    t_max, t_min = tensor.max(), tensor.min()
    
    scale = max(abs(t_max), abs(t_min)) / 127
    
    q_t = np.clip(np.round(tensor / scale), a_min=-127, a_max=127).astype(np.int8)
    dq_t = scale * q_t.astype(np.float32)
     
    return q_t, dq_t, scale

## Part B: Per-Output-Channel Symmetric Quantization

In [4]:
def per_channel_quantize(tensor):
    scale = [np.float32(0.)] * tensor.shape[0]
    q_t = np.ndarray(tensor.shape, dtype=np.int8)
    dq_t = np.ndarray(tensor.shape, dtype=np.float32)
    for oc in range(tensor.shape[0]):
        q_t[oc], dq_t[oc], scale[oc] = per_tensor_quantize(tensor[oc])
    return q_t, dq_t, scale

In [5]:
q_t, dq_t, scale = per_tensor_quantize(conv_weights)


In [6]:
q_pc, dq_pc, scales = per_channel_quantize(conv_weights)

In [ ]:
# display output 
import pandas as pd

pt_mae = np.mean(np.abs(conv_weights - dq_t))
bound_avg = (0.0, 0.0)
table = []
for channel in range(conv_weights.shape[0]):
    bound = (f"{conv_weights[channel].min():.4f}", f"{conv_weights[channel].max():.4f}")
    pc_mae = np.mean(np.abs(conv_weights[channel] - dq_pc[channel]))
    is_better = pt_mae > pc_mae
    table.append((channel, bound, scale, pt_mae, scales[channel],pc_mae, is_better))

# calculate avg for numerical fields
avg_scale = np.mean(scales)
avg_pch_mae = np.mean([np.mean(np.abs(conv_weights[c] - dq_pc[c])) for c in range(conv_weights.shape[0])])

avg_range = (f"({conv_weights.min(axis=(1, 2, 3)).mean():.4f}, {conv_weights.max(axis=(1, 2, 3)).mean():.4f})")

table.append(("AVG", avg_range, scale, pt_mae, avg_scale, avg_pch_mae, pt_mae > avg_pch_mae))
df = pd.DataFrame(table, columns=["Channel", "Range", "Per-Tensor Scale", "Per-Tensor MAE", "Per-Ch Scale", "Per-Ch MAE", "Better"])

print(df.to_string(index=False))

Channel               Range  Per-Tensor Scale  Per-Tensor MAE  Per-Ch Scale  Per-Ch MAE  Better
      0   (-0.1913, 0.1579)          0.303365        0.071143      0.001507    0.000326    True
      1   (-0.9798, 0.9261)          0.303365        0.071143      0.007715    0.001661    True
      2   (-2.6197, 1.5646)          0.303365        0.071143      0.020628    0.005373    True
      3   (-1.4635, 1.8862)          0.303365        0.071143      0.014852    0.003731    True
      4   (-1.9188, 2.4632)          0.303365        0.071143      0.019396    0.003995    True
      5   (-1.6075, 1.8658)          0.303365        0.071143      0.014691    0.003901    True
      6  (-5.3545, 13.6008)          0.303365        0.071143      0.107093    0.027714    True
      7 (-15.1485, 38.5273)          0.303365        0.071143      0.303365    0.064038    True
    AVG   (-3.6605, 7.6240)          0.303365        0.071143      0.061156    0.013842    True


## Expected Output

|Channel | Range | Per-Tensor Scale | Per-Tensor MAE | Per-Ch Scale | Per-Ch MAE | Better |
|---|----|---|---|---|---|---|
|0 | (-0.1913, 0.1579) | 0.303365 | 0.071143 | 0.001507 | 0.000326 | True |
|1 | (-0.9798, 0.9261) | 0.303365 | 0.071143 | 0.007715 | 0.001661 | True |
|2 | (-2.6197, 1.5646) | 0.303365 | 0.071143 | 0.020628 | 0.005373 | True |
|3 | (-1.4635, 1.8862) | 0.303365 | 0.071143 | 0.014852 | 0.003731 | True |
|4 | (-1.9188, 2.4632) | 0.303365 | 0.071143 | 0.019396 | 0.003995 | True |
|5 | (-1.6075, 1.8658) | 0.303365 | 0.071143 | 0.014691 | 0.003901 | True |
|6 | (-5.3545, 13.6008) | 0.303365 | 0.071143 | 0.107093 | 0.027714 | True |
|7 | (-15.1485, 38.5273) | 0.303365 | 0.071143 | 0.303365 | 0.064038 | True |
|AVG |(-3.6605, 7.6240) | 0.303365 | 0.071143 | 0.061156 | 0.013842 | True |


## Observation:
- For a imbalanced weight (like ones in regular convolution layers) Per channel quantization Yields better results compared to the regular Per-Tensor Quantization technique.
- But this inturn takes more memory and compute to store and process as it has to save channel for each channel separately.